In [1]:
import pandas as pd

claimant_wide = pd.read_csv("London_Borough_Claimant_Count_WIDE.csv")

claimant_long = claimant_wide.melt(id_vars="Borough", var_name="MonthStr", value_name="Claimants")
claimant_long["Date"] = pd.to_datetime(claimant_long["MonthStr"], format="%b-%Y")
claimant_long = claimant_long.drop(columns="MonthStr")

claimant_london = claimant_long.groupby("Date")["Claimants"].sum().reset_index()

print(claimant_london.shape)
print(claimant_london.head())
print(claimant_london.tail())

(198, 2)
        Date  Claimants
0 2010-01-01     225930
1 2010-02-01     229080
2 2010-03-01     227225
3 2010-04-01     222605
4 2010-05-01     218600
          Date  Claimants
193 2026-02-01     344870
194 2026-03-01     344010
195 2026-04-01     341605
196 2026-05-01     347495
197 2026-06-01     353225


In [3]:
crime = pd.read_csv("cleaned_crime_long_through_june2026.csv")
crime["Date"] = pd.to_datetime(crime["Date"])
london_crime = crime.groupby("Date")["CrimeCount"].sum().reset_index()

merged = london_crime.merge(claimant_london, on="Date", how="inner")
print(merged.shape)
merged.tail()

(195, 3)


,Date,CrimeCount,Claimants
190,2026-02-01,67469,344870
191,2026-03-01,75137,344010
192,2026-04-01,73990,341605
193,2026-05-01,78652,347495
194,2026-06-01,79894,353225


In [8]:
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX

claimant_wide = pd.read_csv("London_Borough_Claimant_Count_WIDE.csv")
claimant_long = claimant_wide.melt(id_vars="Borough", var_name="MonthStr", value_name="Claimants")
claimant_long["Date"] = pd.to_datetime(claimant_long["MonthStr"], format="%b-%Y")
claimant_long = claimant_long.drop(columns="MonthStr")
claimant_london = claimant_long.groupby("Date")["Claimants"].sum().reset_index()

crime = pd.read_csv("data/cleaned_crime_long_through_june2026.csv")
crime["Date"] = pd.to_datetime(crime["Date"])
london_crime = crime.groupby("Date")["CrimeCount"].sum().reset_index()

merged = london_crime.merge(claimant_london, on="Date", how="inner")
merged = merged.sort_values("Date").reset_index(drop=True)

print("Merged shape:", merged.shape)

cutoff = pd.Timestamp("2026-02-28")
start_test = pd.Timestamp("2026-03-01")
end_test = pd.Timestamp("2026-06-30")

train = merged[merged["Date"] <= cutoff].set_index("Date")
test = merged[(merged["Date"] >= start_test) & (merged["Date"] <= end_test)].set_index("Date")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

y_train = train["CrimeCount"]
y_test = test["CrimeCount"]
exog_train = train[["Claimants"]]
exog_test = test[["Claimants"]]

model_baseline = SARIMAX(y_train, order=(1,1,0), seasonal_order=(1,1,0,12),
                          enforce_stationarity=False, enforce_invertibility=False)
fit_baseline = model_baseline.fit(disp=False)
forecast_baseline = fit_baseline.forecast(steps=4)

mae_baseline = np.mean(np.abs(y_test.values - forecast_baseline.values))

model_exog = SARIMAX(y_train, exog=exog_train, order=(1,1,0), seasonal_order=(1,1,0,12),
                      enforce_stationarity=False, enforce_invertibility=False)
fit_exog = model_exog.fit(disp=False)
forecast_exog = fit_exog.forecast(steps=4, exog=exog_test)

mae_exog = np.mean(np.abs(y_test.values - forecast_exog.values))

print("\nBaseline SARIMA (no exog) MAE:", round(mae_baseline, 1))
print("SARIMA + Claimant Count MAE:", round(mae_exog, 1))

comparison = pd.DataFrame({
    "Month": test.index.strftime("%Y-%m"),
    "Actual": y_test.values,
    "Forecast_Baseline": forecast_baseline.round(0).values,
    "Forecast_With_Claimants": forecast_exog.round(0).values
})
print("\n", comparison)

Merged shape: (195, 3)
Train shape: (191, 2)
Test shape: (4, 2)

Baseline SARIMA (no exog) MAE: 4235.4
SARIMA + Claimant Count MAE: 4237.1

      Month  Actual  Forecast_Baseline  Forecast_With_Claimants
0  2026-03   75137            71670.0                  71671.0
1  2026-04   73990            70293.0                  70291.0
2  2026-05   78652            74543.0                  74539.0
3  2026-06   79894            74225.0                  74224.0


/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


In [9]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

claimant_ts = claimant_london.set_index("Date")["Claimants"]

model_claimants = SARIMAX(claimant_ts, order=(1,1,0), seasonal_order=(1,1,0,12),
                           enforce_stationarity=False, enforce_invertibility=False)
fit_claimants = model_claimants.fit(disp=False)

forecast_claimants = fit_claimants.forecast(steps=5)

forecast_df = pd.DataFrame({
    "Month": forecast_claimants.index.strftime("%Y-%m"),
    "Forecast_Claimants": forecast_claimants.round(0).values
})
print(forecast_df)

     Month  Forecast_Claimants
0  2026-07            366582.0
1  2026-08            361364.0
2  2026-09            363064.0
3  2026-10            363574.0
4  2026-11            358488.0


/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/opt/conda/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


In [10]:
forecast_claimants = fit_claimants.forecast(steps=6)

forecast_df = pd.DataFrame({
    "Month": forecast_claimants.index.strftime("%Y-%m"),
    "Forecast_Claimants": forecast_claimants.round(0).values
})

forecast_df = forecast_df[forecast_df["Month"] != "2026-07"].reset_index(drop=True)
print(forecast_df)


     Month  Forecast_Claimants
0  2026-08            361364.0
1  2026-09            363064.0
2  2026-10            363574.0
3  2026-11            358488.0
4  2026-12            356051.0
